# Market and News Data Collection

Collect historical market data and current financial news for downstream research. Credentials are read from the runtime environment and are never stored in this notebook.

In [1]:
import os
from pathlib import Path

import pandas as pd
import requests
import yfinance as yf
from dotenv import load_dotenv

project_root = Path.cwd()
if not (project_root / "pyproject.toml").exists():
    project_root = (Path.cwd() / "..").resolve()
load_dotenv(project_root / ".env")

True

In [2]:
stock_ticker = os.getenv("STOCK_TICKER", "AAPL").strip().upper()
data_period = os.getenv("DATA_PERIOD", "1y").strip()

if not stock_ticker or not data_period:
    raise ValueError("STOCK_TICKER and DATA_PERIOD are required.")

print(f"Ticker: {stock_ticker}")
print(f"Period: {data_period}")

Ticker: AAPL
Period: 1y


In [3]:
stock = yf.Ticker(stock_ticker)
stock_df = stock.history(period=data_period).reset_index()
if stock_df.empty:
    raise ValueError(f"No market data returned for {stock_ticker}.")

try:
    company_name = stock.info.get("longName", stock_ticker)
except Exception:
    company_name = stock_ticker

display(stock_df.head())
print(f"Market rows: {len(stock_df):,}")

,Date,Open,High,Low,Close,Volume,Dividends,Stock Splits
0,2025-10-10 00:00:00-04:00,254.001734,255.436437,243.101995,244.367325,61999100,0.0,0.0
1,2025-10-13 00:00:00-04:00,248.462191,248.771048,244.656243,246.748520,38142900,0.0,0.0
2,2025-10-14 00:00:00-04:00,245.692416,247.934135,243.799400,246.858109,35478000,0.0,0.0
3,2025-10-15 00:00:00-04:00,248.571790,250.893216,246.559220,248.422333,33893600,0.0,0.0
4,2025-10-16 00:00:00-04:00,247.336335,248.123421,244.227823,246.539276,39777000,0.0,0.0


Market rows: 251


In [4]:
news_api_key = os.getenv("NEWS_API_KEY")

if not news_api_key:
    raise RuntimeError(
        "NEWS_API_KEY is missing. Add NEWS_API_KEY=... to the project "
        ".env file, then restart the notebook kernel."
    )

response = requests.get(
    "https://newsapi.org/v2/everything",
    params={
        "q": stock_ticker,
        "apiKey": news_api_key,
        "language": "en",
        "pageSize": 50,
        "sortBy": "relevancy",
    },
    timeout=30,
)
response.raise_for_status()
news_df = pd.DataFrame(response.json().get("articles", []))
display(news_df.head())
print(f"News rows: {len(news_df):,}")

,source,author,title,description,url,urlToImage,publishedAt,content
0,"{'id': None, 'name': 'MacRumors'}",Joe Rossignol,"New Mac Mini Has Two Storage Downsides, Includ...","As confirmed by the website ""Expand Mac mini,""...",https://www.macrumors.com/2026/09/22/new-mac-m...,https://images.macrumors.com/t/Nn9fuxpIL2pwS6U...,2026-09-23T03:23:24Z,"As confirmed by the website ""Expand Mac mini,""..."
1,"{'id': None, 'name': 'Yahoo Entertainment'}","Brooke DiPalma, Michael B. Kelley","Paramount-Warner Bros. finally merger closes, ...",The $110 billion merger of Paramount Skydance ...,https://finance.yahoo.com/markets/stocks/artic...,https://s.yimg.com/lo/mysterio/api/ea08c179d8a...,2026-10-06T13:03:29Z,The $110 billion merger of Paramount Skydance ...
2,"{'id': None, 'name': 'Yahoo Entertainment'}","Brooke DiPalma, Michael B. Kelley","Paramount-Warner Bros. finally merger closes, ...",The $110 billion merger of Paramount Skydance ...,https://finance.yahoo.com/markets/stocks/artic...,https://s.yimg.com/lo/mysterio/api/ea08c179d8a...,2026-10-06T13:03:29Z,The $110 billion merger of Paramount Skydance ...
3,"{'id': None, 'name': 'Yahoo Entertainment'}",Ben Werschkul,Trump and Xi meet in D.C. this week with tarif...,President Trump meets later this week with Chi...,https://finance.yahoo.com/markets/article/trum...,https://s.yimg.com/lo/mysterio/api/5e1eb589956...,2026-09-21T10:00:00Z,President Trump meets later this week with Chi...
4,"{'id': None, 'name': 'Barchart.com'}","Mark R. Hake, CFA","Large, Unusual Volume in Long-Dated Apple Put ...",Heavy volume in Apple puts expiring in one yea...,https://www.barchart.com/story/news/4639962/la...,https://s.yimg.com/lo/mysterio/api/b7f193a0121...,2026-09-16T17:30:02Z,Apple store and shoppers - by PhillDanze via i...


News rows: 50


In [5]:
output_dir = project_root / "data" / "processed"
output_dir.mkdir(parents=True, exist_ok=True)
stock_df.to_csv(output_dir / f"market_{stock_ticker}.csv", index=False)
news_df.to_json(output_dir / f"news_{stock_ticker}.json", orient="records", date_format="iso")
print(f"Saved reusable outputs in repository root: {output_dir}")

Saved reusable outputs in repository root: /Users/piyushpravin/Library/CloudStorage/OneDrive-Personal/USD MS AAI/AAI-520-Project/data/processed
